In [13]:
import pandas as pd

file_path = "/Users/sabirinomar/Politics-Network/data/combined_articles.csv"

combined = pd.read_csv(file_path, sep=";")

print(combined.shape)
print(combined.columns.tolist())
print(combined.head())

(249, 8)
['url', 'dato', 'sektion', 'titel', 'tekst', 'kilde', 'antal_politikere', 'politikere ']
                                                 url        dato  sektion  \
0  https://www.dr.dk/nyheder/politik/14-hurtige-h...  2022-10-05  politik   
1  https://www.dr.dk/nyheder/politik/cordsens-ana...  2022-10-05  politik   
2  https://www.dr.dk/nyheder/politik/folketingsva...  2022-10-05  politik   
3  https://www.dr.dk/nyheder/politik/folketingsva...  2022-10-05  politik   
4  https://www.dr.dk/nyheder/politik/folketingsva...  2022-10-05  politik   

                                               titel  \
0  14 hurtige: Her er det, som partilederne mener...   
1  Cordsens analyse: Sådan er kampformen hos de 1...   
2  I kommer med 'de lette løsningers parade', lød...   
3  Kæmpe kursskifte: Mette Frederiksen går nu eft...   
4  Partiledere er klar til valgkamp: 'Nu kører Pa...   

                                               tekst kilde  antal_politikere  \
0  Hvad er det vigtigs

In [14]:
#use da for Danish language model
import spacy


nlp = spacy.load("da_core_news_sm") 

def preprocess_text(text):
    if pd.isna(text):
        return []
    doc = nlp(str(text))

    tokens = [
        token.lemma_.lower() 
        for token in doc 
        if not token.is_stop and not token.is_punct and not token.is_space and token.lemma and len(token.text) > 2
    ]
    return tokens

#df['tokens'] = df['body'].astype(str).apply(preprocess_text)
combined['tokens'] = combined['tekst'].astype(str).apply(preprocess_text)

#sample_text = "Dette er en lille testtekst til at se, hvordan preprocessing fungerer, da jeg gik"
#print("Original:", sample_text)
#print("Processed Tokens:", preprocess_text(sample_text))

print("Sample tokens from the first article:", combined['tokens'].iloc[0]) 

Sample tokens from the first article: ['vigtig', 'valgkamp', 'spørgsmål', 'indlede', 'valgkamp', 'første', 'partilederdebat', 'christiansborg', 'samtlige', 'frontfigure', 'opstillingsberette', 'parti', 'sekund', 'svare', 'svare', 'socialdemokrati', 'statsminister', 'mette', 'frederiksen', 'vigtig', 'danskernes', 'tryghed', 'socialdemokrati', 'statsminister', 'mette', 'frederiksen', 'vigtig', 'danskernes', 'tryghed', 'venstre', 'jakob', 'ellemann-jensen', 'vigtigst', 'styr', 'økonomi', 'venstre', 'jakob', 'ellemann-jensen', 'vigtigst', 'styr', 'økonomi', 'løs', 'udfordring', 'klimaområd', 'bekæmp', 'mistrivsel', 'barn', 'ung', 'vigtigst', 'pia', 'olsen', 'dyhr', 'løs', 'udfordring', 'klimaområd', 'bekæmp', 'mistrivsel', 'barn', 'ung', 'vigtigst', 'pia', 'olsen', 'dyhr', 'radikal', 'venstre', 'sofie', 'carsten', 'nielsen', 'vigtig', 'løse', 'aktuel', 'krise', 'samarbejde', 'tværs', 'blokk', 'radikal', 'venstre', 'sofie', 'carsten', 'nielsen', 'vigtig', 'løse', 'aktuel', 'krise', 'samarbe

In [15]:
# Extract Top Keywords across the entire corpus
all_tokens = [token for sublist in combined['tokens'] for token in sublist]
counts = Counter(all_tokens)

print("Top 150 Most Common Tokens:")
for token, count in counts.most_common(150):
    print(f"{token}: {count}")

# Select the top N most frequent words as your keyword vocabulary (e.g., top 150)
top_keywords = set([word for word, freq in counts.most_common(300)])
print(f"Top keywords extracted: {len(top_keywords)} keywords selected.")

NameError: name 'Counter' is not defined

In [ ]:
from pathlib import Path

# Create output directory
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Save tokenized dataset as CSV
output_file = output_dir / "tokenized_articles.csv"
combined.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"Saved {len(combined)} articles to: {output_file}")

Saved 249 articles to: data/processed/tokenized_articles.csv


## 1. tokenize parti citater 

In [16]:
import pandas as pd
citater = pd.read_csv("/Users/sabirinomar/Politics-Network/data/citater_pr_politiker.csv")

citater["tokens"] = citater["citat"].apply(preprocess_text)

print(citater[["politiker", "parti", "citat", "tokens"]].head())

       politiker parti                                              citat  \
0  Aaja Chemnitz    IA  Han brugte det jo faktisk til at fremme sin eg...   
1  Aaja Chemnitz    IA  Det er i hvert fald ikke blevet taget særligt ...   
2  Aaja Chemnitz    IA  Hvis Pape bliver statsminister, vil et frugtba...   
3  Aaja Chemnitz    IA  Muligheden for et samarbejde er fortsat yderst...   
4  Aaja Chemnitz    IA  Det har ikke gjort noget bedre for ham, når de...   

                                              tokens  
0  [bruge, faktisk, fremme, sin, sag, valgtrick, ...  
1  [hver, fald, blive, tage, særligt, godt, grønl...  
2  [pape, statsminister, frugtbart, godt, samarbe...  
3  [mulighed, samarbejde, fortsat, yderst, vanske...  
4                                 [bedre, når, måde]  


In [18]:
for _, row in citater.head(10).iterrows():
    print("Politiker:", row["politiker"])
    print("Original:", row["citat"])
    print("Tokens:", row["tokens"])
    print("-" * 60)

Politiker: Aaja Chemnitz
Original: Han brugte det jo faktisk til at fremme sin egen sag. Det var nok lidt mere et valgtrick – også fordi det kom meget sent i forløbet, siger Aaja Chemnitz, der også udtaler, at hun ikke er alene med den holdning.
Tokens: ['bruge', 'faktisk', 'fremme', 'sin', 'sag', 'valgtrick', 'sent', 'forløb', 'sige', 'aaja', 'chemnitz', 'udtaler', 'holdning']
------------------------------------------------------------
Politiker: Aaja Chemnitz
Original: Det er i hvert fald ikke blevet taget særligt godt imod her i Grønland. Det kan jeg se på sociale medier og høre, når jeg snakker med grønlændere, siger hun.
Tokens: ['hver', 'fald', 'blive', 'tage', 'særligt', 'godt', 'grønland', 'social', 'medie', 'høre', 'når', 'snakke', 'grønlænder', 'sige']
------------------------------------------------------------
Politiker: Aaja Chemnitz
Original: Hvis Pape bliver statsminister, vil et frugtbart og godt samarbejde være meget begrænset.
Tokens: ['pape', 'statsminister', 'frugt

In [19]:
from pathlib import Path

output_dir = Path("data/")
output_dir.mkdir(parents=True, exist_ok=True)

citater.to_csv(
    output_dir / "tokenized_citater.csv",
    index=False,
    encoding="utf-8-sig"
)